## Single Run Computation: The Adult's Game
A single run can be depicted as a single adult throwing a pebble in a random direction, then walking over to repeat the action for a number of times (*trials*), forming a chain of landings with each dependent on the previous location. 

In [1]:
import random

x, y = 1.0, 1.0
delta = 0.1
n_trials = 4000
n_hits = 0
for i in range(n_trials):
    del_x, del_y = random.uniform(-delta, delta), random.uniform(-delta, delta)
    if abs(x + del_x) < 1.0 and abs(y + del_y) < 1.0:
        x, y = x + del_x, y + del_y
    if x**2 + y**2 < 1.0: n_hits += 1
print (4.0 * n_hits / n_trials)

3.425


## Multirun Simulation: Multiple Chains of Events
Now, let's increase the number of adults playing on the heliport, giving us a richer dataset.

In [9]:
import random
import statistics

def estimate_pi(N, delta): 
    x, y = 1.0, 1.0
    n_hits = 0
    for i in range(N):
        del_x, del_y = random.uniform(-delta, delta), random.uniform(-delta, delta)
        if abs(x + del_x) < 1.0 and abs(y + del_y) < 1.0:
            x, y = x + del_x, y + del_y
        if x**2 + y**2 < 1.0: n_hits += 1
    return 4.0 *n_hits / N

def main():
    n_runs = 1_000
    n_trials = 400_000
    delta = 0.1
    estimates = [estimate_pi(n_trials, delta) for _ in range(n_runs)]

    mean_est = statistics.mean(estimates)
    std_est = statistics.pstdev(estimates)
    #std is rounded to 6 decimal places to match the output of the original code
    print(f"Run: {n_runs}, Trials: {n_trials}, Mean: {mean_est:.6f}, Std: {std_est:.6f}")

if __name__ == "__main__":
    main()

Run: 1000, Trials: 400000, Mean: 3.140450, Std: 0.022628


### NumPy or Numba Optimisation?: Markov Chains
As the name implies, it is not probable for NumPy to vectorise and generate all the samples in a Markov Chain as each sample of $n>0$ (aside from the starting sample) depends on the previous sample. However, it is possible for NumPy to vectorise across independent chains instead of vectorising across time.

Furthermore, an `if` loop for $x^2 + y^2 \leq 1$ can't be applied here as each vectorised chain samples are an array instead of individual values unlike in DMC. Hence, we will need to adjust the code for the chain optimisation. 


In [ ]:
import numpy as np

def estimate_pi(n_trials, n_runs,delta):
    x, y = np.zeros(n_runs), np.zeros(n_runs)
    n_hits = np.zeros(n_runs)
    
    for _ in range(n_trials):
        del_x = np.random.uniform(-delta, delta, n_runs)
        del_y = np.random.uniform(-delta, delta, n_runs)

        X = x + del_x
        Y = y + del_y   
        
        accept = (np.abs(X) < 1.0) & (np.abs(Y) < 1.0)
        # Update positions only if they remain within the unit square
        x[accept] = X[accept]
        y[accept] = Y[accept]
        
        n_hits += x**2 + y**2 <= 1.0

    return 4.0 * n_hits / n_trials

def main():
    n_runs = 1_000
    n_trials = 400_000
    estimates = estimate_pi(n_trials, n_runs, 0.1)
    
    mean_est = np.mean(estimates)
    std_est = np.std(estimates)
    sem = std_est / np.sqrt(n_runs)
    return mean_est, std_est, sem, n_runs, n_trials

if __name__ == "__main__":
    mean_est, std_est, sem, n_runs, n_trials = main()
    print(f"Run: {n_runs}, Trials: {n_trials}, Average value: {mean_est:.6f}, Standard deviation: {std_est:.6f}, Standard error: {sem:.6f}")

Run: 1000, Trials: 400000, Average value: 3.141783, Standard deviation: 0.022149, Standard error: 0.000700


The optimisation of individual chains can significantly reduce the algorithm's runtime by ~10 fold.

However, Numba can be used to further optimise MCMC algorithms by compling the loop into a single operation instead of a repeating iteration loop.

In [24]:
import numpy as np
from numba import njit

@njit
def estimate_pi(n_trials, delta):
    x, y = 0, 0
    n_hits = 0

    for _ in range(n_trials):
        del_x = np.random.uniform(-delta, delta)
        del_y = np.random.uniform(-delta, delta)
        if abs(x + del_x) < 1.0 and abs(y + del_y) < 1.0:
            x, y = x + del_x, y + del_y
        if x**2 + y**2 < 1.0: n_hits += 1
    
    return 4.0 * n_hits / n_trials

def main():
    n_runs = 1_000
    n_trials = 400_000
    estimates = [estimate_pi(n_trials, 0.1) for _ in range(n_runs)]
    
    mean_est = np.mean(estimates)
    std_est = np.std(estimates)
    sem = std_est / np.sqrt(n_runs)
    return mean_est, std_est, sem, n_runs, n_trials

if __name__ == "__main__":
    mean_est, std_est, sem, n_runs, n_trials = main()
    print(f"Run: {n_runs}, Trials: {n_trials}, Average value: {mean_est:.6f}, Standard deviation: {std_est:.6f}, Standard error: {sem:.6f}")
        

Run: 1000, Trials: 400000, Average value: 3.141754, Standard deviation: 0.022174, Standard error: 0.000701
